# Working with Hugging Face Local Models

Learn how to download models from Hugging Face Hub, store them locally, and load them from a local directory.

`BAAI/bge-m3` is an example model used in AI applications, such as [ChatQuran: A Multilingual RAG Application](https://aviosit.com/projects/chatquran), for generating text embeddings.

This tutorial covers model downloading and loading only. Embedding generation is covered in the next tutorial: [How to Embed Texts for AI Applications](https://aviosit.com/tutorials/embedding-texts-for-ai-applications).

## 1. Set up the environment

Install the required dependencies using `uv` from the repository root: `uv sync`

In [1]:
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

/Users/muhajirin/project/aviosit-tutorials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure the model path

Set the model repository ID and local directory path. The model will be stored in `models/bge-m3` at the repository root.

In [2]:
REPO_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the repository root containing pyproject.toml."
    )

MODEL_REPOSITORY = "BAAI/bge-m3"
MODEL_PATH = REPO_ROOT / "models" / "bge-m3"

MODEL_PATH.mkdir(parents=True, exist_ok=True)

print(f"Model repository : {MODEL_REPOSITORY}")
print(f"Local directory  : {MODEL_PATH}")

Model repository : BAAI/bge-m3
Local directory  : /Users/muhajirin/project/aviosit-tutorials/models/bge-m3


## 3. Download and inspect the model

Use `snapshot_download()` to download the model repository to the configured local directory. If the required model files are already present, skip the download. Otherwise, download the model files. Ensure that network access and sufficient disk space are available for the initial download.

In [3]:
required_files = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
]

model_ready = (
    all((MODEL_PATH / filename).is_file() for filename in required_files)
    and (
        any(MODEL_PATH.glob("*.safetensors"))
        or any(MODEL_PATH.glob("pytorch_model*.bin"))
    )
)

if model_ready:
    print(f"Model files already exist at: {MODEL_PATH.relative_to(REPO_ROOT)}")
else:
    print(f"Downloading {MODEL_REPOSITORY}...")
    snapshot_download(
        repo_id=MODEL_REPOSITORY,
        local_dir=MODEL_PATH,
    )

model_files = sorted(
    path
    for path in MODEL_PATH.rglob("*")
    if path.is_file()
    and ".cache" not in path.parts
    and path.name not in {".DS_Store", ".gitkeep"}
)

if not model_files:
    raise RuntimeError(f"No model files found in {MODEL_PATH}")

print(f"Model files: {len(model_files)}")

for path in model_files[:15]:
    print(path.relative_to(MODEL_PATH))

if len(model_files) > 15:
    print(f"... and {len(model_files) - 15} more files")


Model files already exist at: models/bge-m3
Model files: 29
.gitattributes
1_Pooling/config.json
README.md
colbert_linear.pt
config.json
config_sentence_transformers.json
imgs/bm25.jpg
imgs/long.jpg
imgs/miracl.jpg
imgs/mkqa.jpg
imgs/nqa.jpg
imgs/others.webp
long.jpg
modules.json
onnx/Constant_7_attr__value
... and 14 more files


## 4. Load the model locally

Load the tokenizer and model from the downloaded directory using Transformers. Set `local_files_only=True` to prevent downloading missing files from Hugging Face Hub during loading.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model = AutoModel.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model.eval()

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Model: {model.__class__.__name__}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49071.60it/s]

Tokenizer: XLMRobertaTokenizer
Model: XLMRobertaModel


## 5. Verify the model

Verify that the model has loaded correctly by checking its configuration and hidden size. This tutorial focuses on downloading and loading models; generating embeddings is covered in the next tutorial.

In [5]:
print("Model type:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)

assert model.config.hidden_size == 1024, (
    "Unexpected hidden size. Check that MODEL_PATH points to BAAI/bge-m3."
)

Model type: xlm-roberta
Hidden size: 1024


### Next tutorial

Continue with [How to Embed Texts for AI Applications](https://aviosit.com/tutorials/embedding-texts-for-ai-applications) to learn how to generate text embeddings using BGE-M3.